# Course 6: From Beginner to an AI Research Lab
## Session 12: Getting in

**Goal:** build an application tracker across several routes, check a cold email against three tests, and run a timed mock interview.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a GPU, a download or an account. Cells that are worksheets hold example rows for you to replace with your own.

### 1. The tracker
At least eight targets across at least two routes, with deadlines and what each one needs. Replace the example rows. Dates below are placeholders: check every deadline on the official page.

In [ ]:
import pandas as pd
from datetime import date
tracker = pd.DataFrame([
    {"target": "University A PhD", "route": "PhD", "deadline": "2026-12-15", "needs": "SoP, 3 letters, CV", "status": "drafting"},
    {"target": "Prof. B's lab", "route": "Remote RA", "deadline": "rolling", "needs": "tailored email", "status": "sent"},
    {"target": "Open community C", "route": "Open", "deadline": "none", "needs": "first contribution", "status": "joined"},
    {"target": "Fellowship D", "route": "Fellowship", "deadline": "check", "needs": "statement, work test", "status": "planned"},
])
def review(df, today):
    out = df.copy(); d = pd.to_datetime(out.deadline, errors="coerce")
    out["days_left"] = (d - pd.Timestamp(today)).dt.days
    problems = []
    if len(out) < 8: problems.append(f"only {len(out)} targets: aim for at least 8")
    if out.route.nunique() < 2: problems.append("only one route")
    if (out.deadline == "check").any(): problems.append("some deadlines are not looked up yet")
    soon = out[(out.days_left < 28) & out.needs.str.contains("letter", case=False)]
    if len(soon): problems.append(f"letters needed within 4 weeks for: {list(soon.target)}. Ask today.")
    return out.sort_values("days_left"), problems
view, problems = review(tracker, date(2026, 11, 20))
print(view.to_string(index=False)); print("to fix:", problems)
assert any("letters" in p for p in problems)

Ask for letters at least four weeks ahead, and send each writer your CV, your statement and a short list of what you did with them.

### 2. The one-page CV: order and wording
Education; research experience with the capstone first and links to the paper and code; other projects; skills; awards. Lead each item with what you found, not what you were assigned.

In [ ]:
WEAK = ["responsible for", "worked on", "helped with", "participated in", "assisted", "was involved in", "coursework"]
def cv_line_check(line):
    notes = []
    if any(w in line.lower() for w in WEAK): notes.append("leads with a duty, not a result")
    if not any(ch.isdigit() for ch in line): notes.append("no number")
    if "http" not in line and "github" not in line.lower(): notes.append("no link to evidence")
    return notes or ["good"]
for line in ["Worked on a transformer project for the course.",
             "Reproduced Table 2 of <paper> within 0.4 points over 5 seeds; found the gain disappears without warm-up. Code: github.com/<you>/<repo>"]:
    print(line, "\n   ->", cv_line_check(line))

### 3. The cold email: three tests
Does it name a paper, show your work with one link, and make one concrete ask? And is it short enough to read on a phone?

In [ ]:
import re
def email_check(text):
    words = len(text.split())
    return {"words": words, "short_enough (<=150)": words <= 150,
            "names_a_specific_paper": bool(re.search(r"your (paper|work|preprint) [\"'“<]", text.lower())) or "arxiv" in text.lower(),
            "one_link_to_my_work": len(re.findall(r"https?://", text)) == 1,
            "one_concrete_ask": text.count("?") == 1}
good = """Dear Dr. <Name>,
I reproduced Table 2 of your paper "<Title>" on a smaller model and found the gain holds at 10M parameters but shrinks without warm-up (5 seeds; report and code: https://github.com/<you>/<repo>).
I am a final-year CS student at <University>. Would you have 20 minutes in the next month to tell me whether the warm-up effect is worth a short workshop paper?
Thank you,
<Name>"""
bad = "Dear Professor, I am very passionate about AI and deep learning and I have always dreamed of working in your esteemed lab. " * 4 + "Can I join? Do you have funding? Can you reply soon?"
print("good:", email_check(good)); print("bad :", email_check(bad))
assert all(v for k, v in email_check(good).items() if k != "words") and not email_check(bad)["one_concrete_ask"]

### 4. The research statement and the statement of purpose
One page: what you did and found, what you want to study next, and why this place. For a program, name two possible advisors and tie each to your question with a specific paper of theirs.

In [ ]:
def sop_paragraph_check(text, advisors):
    named = [a for a in advisors if a.lower() in text.lower()]
    return {"advisors_named": named, "at_least_two": len(named) >= 2, "mentions_a_paper_or_result": any(w in text.lower() for w in ["paper", "showed", "found", "result"]),
            "generic_phrases": [p for p in ["world-class", "prestigious", "since childhood", "passion"] if p in text.lower()]}
print(sop_paragraph_check("I want to work with Prof. Rahman, whose paper on X showed Y, and with Prof. Chen on Z.", ["Rahman", "Chen"]))

### 5. Mock interview, 30 minutes each way
A 5-minute project talk, 10 minutes of fundamentals, 10 minutes of coding on a shared screen, and 5 minutes of feedback.

In [ ]:
import random
SCHEDULE = [("Project talk", 5), ("Fundamentals", 10), ("Coding on a shared screen", 10), ("Feedback", 5)]
t = 0
for part, minutes in SCHEDULE: print(f"{t:02d}:00 - {t + minutes:02d}:00   {part}"); t += minutes
assert t == 30
FUNDAMENTALS = ["What is overfitting, and how do you detect it?", "Why do we need a validation set as well as a test set?", "What does the learning rate control?",
                "Explain backpropagation in two sentences.", "What does attention compute?", "How would you check that a gradient is correct?",
                "Your validation loss rises while training loss falls. What do you do?", "How many seeds did you run, and why that many?"]
random.seed(0); print("today's fundamentals:", random.sample(FUNDAMENTALS, 4))
RUBRIC = pd.DataFrame({"criterion": ["main result stated in the first minute", "evidence given with numbers", "limits stated without being asked", "thinks aloud while coding", "tests the code before saying 'done'", "says 'I do not know' and reasons from there"], "score_1_to_5": ""})
print(RUBRIC.to_string(index=False))

The most useful feedback is usually about the project talk. If your partner cannot repeat your main result afterwards, rewrite the talk. The course document has a longer interview question bank, and Course 8 trains each round in depth.

### 6. After every application
Log the date, what you sent and the outcome. After every interview, write down each question within the hour. Rejections are the normal case on every route; the tracker is what turns them into a next step.

### TODO
1. Fill the tracker with at least eight real targets across at least two routes, with deadlines from the official pages.
2. Write one tailored email and have a classmate run the three tests on it.
3. Hold the mock interview in both directions and hand in the two rubrics.